## melt — 宽表变长表

把横向的列名融化成竖着排的值。三个核心参数：

| 参数 | 作用 |
|---|---|
| `id_vars` | 哪些列不融化（保留为列） |
| `var_name` | 融掉后的"列名列"叫什么 |
| `value_name` | 融掉后的"数值列"叫什么 |

```python
# 宽表：城市名是列头
wide.melt(id_vars='产品', var_name='城市', value_name='销量')
# → 城市列的所有名变成一列的值，销量填进去
```

> melt 是数据清洗里的"旋转刀"——信息藏在列名里时，用它拽出来变成一行行数据。


In [ ]:
import pandas as pd

In [ ]:
wide = pd.DataFrame({
    '产品': ['鼠标','键盘','耳机'],
    '北京': [80, 50, 30],
    '上海': [90, 55, 35],
    '广州': [70, 45, 25],
    '深圳': [85, 48, 28]
})
print(wide)

In [ ]:
print(wide.T)

In [ ]:
#宽表转换(列更多的表)
sales1 = pd.melt(wide,id_vars=['产品'],var_name='城市',value_name='销量',ignore_index=False) #将横着铺开的列名放在一列中的值
sales2 = pd.melt(wide,id_vars=['产品'],var_name='城市',value_name='销量')
#id_vars可保留列不参与融化，依旧为列
#var_name给融化出来的那列起名字
#value_name是给融化后存数值的那列起名字
#ignore_index=False则保留原本索引，行数变多则不断重复
print(sales1)
print(sales1.sort_values('销量'))
print(sales2.sort_values('销量'))

## pivot — 长表变宽表

melt 的逆操作。把"某一列"的值展开为列头。

```python
sales.pivot(index='产品', columns='月份', values='销量')
# → 月份从值变成列名，销量填进对应格子
```

| pivot 参数 | melt 对应 |
|---|---|
| `index` | `id_vars` |
| `columns` | `var_name` |
| `values` | `value_name` |

> **`pivot_table`** 是强化版——当 index+columns 不唯一时（有重复组合），能用 `aggfunc` 聚合。


In [ ]:
#长表转宽表
sales = pd.DataFrame({
    '月份':  ['1月','1月','1月', '2月','2月','2月', '3月','3月','3月'],
    '产品':  ['鼠标','键盘','耳机', '鼠标','键盘','耳机', '鼠标','键盘','耳机'],
    '销量':  [120, 85, 60, 135, 90, 55, 150, 95, 70]
})
print(sales)
sales0 = pd.pivot(sales,index=['产品'],columns=['月份'],values=['销量']) #把一列里的值展开成多列名
#index确定保留谁为行索引
#columns确定哪列的值变成列名
#values确定哪列的数值填进格子
print(sales0)

In [ ]:
#分列
scores = pd.DataFrame({
    '姓名':   ['张三','张三','张三', '李四','李四','李四', '王五','王五','王五'],
    '科目':   ['语文','数学','英语', '语文','数学','英语', '语文','数学','英语'],
    '成绩':   [88, 92, 85, 76, 68, 80, 95, 89, 91],
    '学期':   ['上','上','下', '上','下','下', '上','上','下']
})
print(scores)
scores['姓'] = scores['姓名'].str[0]
scores['名'] = scores['姓名'].str[1:]
print(scores)

In [ ]:
df = pd.read_csv(r'D:/dsml/data/students.csv')
print(df)
df[['中文名','英文名']] = df.姓名.str.split(' ', expand=True)
#双层 df[['列A', '列B']] 返回的是一个 DataFrame 切片，可以同时对多列赋值
#str.split() 默认返回 Series of lists——每个单元格里装一个 ['James', 'Smith'] 这种列表，没法直接塞进两列。expand=True 让它把列表撑开成 DataFrame——第一个单词一列、第二个单词一列
print(df)
print(df.info())
df['中文名'] = df['中文名'].astype('string')
df['英文名'] = df['英文名'].astype('string')
print(df.dtypes)

## 其他变形工具

```python
# 转置
df.T          # 行列互换

# 排序
df.sort_values('列名')            # 按值排
df.sort_index(ascending=False)    # 按索引排

# 轴向操作
df.sum(axis=0)   # 沿着行方向压扁 → 每列之和（默认）
df.sum(axis=1)   # 沿着列方向压扁 → 每行之和

# groupby 后接聚合
df.groupby('品类').agg({'销量': 'sum', '单价': 'mean'})
```

> **axis 记忆法**：删除时 axis=删的对象，聚合时 axis=压扁的方向。`axis=0` 删行/沿行压扁得列结果。


## stack / unstack — 多级索引变形

比 pivot/melt 更适合多级行列结构。

```python
df.stack()      # 最内层列索引→行索引（表变瘦变长）
df.unstack()    # 最内层行索引→列索引（表变宽变扁）
df.unstack(level=1)  # 指定哪一级拽出来
```

| 函数 | pivot/melt | stack/unstack |
|---|---|---|
| 层数 | 单层行列 | 多级索引 |
| 灵活性 | `var_name`/`value_name` 自定义 | `level` 精确控制 |
| 适用 | 日常透视 | 复杂分组后变形 |

> **记住**：melt 融列、pivot 展列、stack/unstack 互为逆运算。日常 pivot + melt 够用，多级索引才需要 stack/unstack。


In [ ]:
# ===== stack：列索引变成行索引（表变瘦变长）=====
# 从 multi DataFrame 开始
import pandas as pd
idx = pd.MultiIndex.from_product([['北京','上海'], ['Q1','Q2']], names=['城市','季度'])
multi = pd.DataFrame({'收入': [100, 120, 140, 160], '利润': [15, 18, 22, 25]}, index=idx)
print('原始多级索引表:')
print(multi)

# stack：把列（收入/利润）压进行索引最后一层
stacked = multi.stack()
print('\nstack 后:')
print(stacked)
print(f'\n类型: {type(stacked)}  — 变成了 Series，只有一列数值')

# unstack 还原
unstacked = stacked.unstack()
print('\nunstack 还原:')
print(unstacked)


In [ ]:
# ===== unstack 指定 level =====
print('原始 multi:')
print(multi)

# unstack 默认把最内层（季度）拽出来变列
print('\nunstack() 默认 → 季度变列名:')
print(multi.unstack())

# unstack(level=0) 把城市拽出来变列
print('\nunstack(level=0) → 城市变列名:')
print(multi.unstack(level=0))

# unstack(level='季度') 用列名指定也一样
print('\nunstack(level="季度"):')
print(multi.unstack(level='季度'))


In [ ]:
# ===== 实战：从 groupby 到 unstack =====
df = pd.read_csv(r'D:/dsml/data/students.csv')

# groupby 后得到多级索引结果
grouped = df.groupby(['专业', '年级'])['GPA'].mean()
print('groupby 后（多级索引 Series）:')
print(grouped.head(6))

# unstack 把年级拽成列名 → 专业×年级的交叉表
pivot_like = grouped.unstack()
print('\nunstack 后 → 看起来像 pivot_table 的结果:')
print(pivot_like)

# 再 stack 回去
back = pivot_like.stack()
print('\nstack 还原:')
print(back.head(6))
